# as-strided-noncontig-source — ex2: transpose breaks contiguity

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five patterns around tensor memory layout that ramp from reading strides → recognizing transpose breaks contiguity → seeing `.view()` fail on non-contig → fixing it with `.contiguous()` → building a zero-copy sliding-window view via `as_strided`. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `as-strided-noncontig-source`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Strides and contiguity — quick refresher

**Stride** = number of elements to skip in storage to advance one step along that axis.
- A contiguous `(H, W)` tensor has stride `(W, 1)`.
- A contiguous `(B, C, H, W)` tensor has stride `(C*H*W, H*W, W, 1)`.

**Contiguity** = the strides match the row-major layout of the current shape.
- `.T` swaps strides but not data → the result is a view but not contiguous.
- `.view()` requires contiguous input — it never copies.
- `.reshape()` makes a view if possible, copies if not.
- `.contiguous()` forces a row-major copy if the tensor isn't already contiguous.

**`as_strided(size, stride)`** is the lowest-level view constructor — you provide the exact shape and stride pair. Bypasses all safety checks; trust the values you pass.

### Exercise 2 — transpose breaks contiguity

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply `is_contiguous()` to show that `.T` returns a non-contiguous view.
> Keywords: transpose, non-contiguous, view-vs-copy
> ```

**KCs targeted:** `transpose-makes-noncontig`

Implement `ex2_transpose_contig_flags(x)` to return a tuple `(orig_contig, transposed_contig)` — the contiguity flag of the original tensor and of its transpose.

For any rectangular (non-square) contiguous matrix, the transpose IS a view (shares storage), but its strides are reversed — `(1, W)` instead of `(W, 1)` — so it's no longer contiguous in row-major order.

In [ ]:
def ex2_transpose_contig_flags(x: Tensor) -> tuple[bool, bool]:
    """Return (x.is_contiguous(), x.T.is_contiguous())."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(12).reshape(3, 4).float()
    orig, t_contig = ex2_transpose_contig_flags(x)
    assert orig is True, f'fresh reshaped tensor should be contiguous, got {orig}'
    assert t_contig is False, f'transpose of (3,4) should be non-contiguous, got {t_contig}'
    # Sanity: storage is shared (the transpose is a view, not a copy).
    assert x.data_ptr() == x.T.data_ptr(), 'x and x.T should share storage'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_transpose_contig_flags(x: Tensor) -> tuple:
    return (x.is_contiguous(), x.T.is_contiguous())
```

**Why transpose is a view.** `.T` doesn't move data — it just swaps the strides. For `(3, 4)` with strides `(4, 1)`, the transpose has shape `(4, 3)` and strides `(1, 4)` over the SAME storage buffer. The result is logically transposed but the memory pattern no longer matches row-major contiguous layout, hence `is_contiguous() → False`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()